# Lesson 01 · What Is a Category?

Stop asking what mathematical objects are made of. Ask only how they map to one another — and almost everything turns out to live in the arrows.

This notebook collects every Python block from the optional labs and exercises of [the lesson page](../lesson-01-what-is-a-category.html), in page order. Run the cells from top to bottom; later cells use names defined in earlier ones.

Python is optional in this course — the lesson is complete without it. *Generated by `tools/sync_code.py`: edit the lesson page, not this file.*

## §6 — Finite categories by hand

**`set_as_category.py`** — morphisms are functions

In [1]:
def compose(g, f):
    "g after f, for functions stored as dictionaries"
    return {x: g[f[x]] for x in f}

A = {1, 2, 3}
f = {1: 'a', 2: 'b', 3: 'a'}     # f : A -> B,  B = {'a', 'b'}
g = {'a': 'yes', 'b': 'no'}      # g : B -> C,  C = {'yes', 'no'}
h = {'yes': 1, 'no': 0}          # h : C -> D,  D = {0, 1}
id_A = {x: x for x in A}
id_B = {y: y for y in 'ab'}

print(compose(g, f))             # the composite g . f : A -> C

# identity law, on both sides
assert compose(f, id_A) == f == compose(id_B, f)
# associativity:  h . (g . f)  ==  (h . g) . f
assert compose(h, compose(g, f)) == compose(compose(h, g), f)
print("laws ok")

{1: 'yes', 2: 'no', 3: 'yes'}
laws ok


**`finite_category.py`** — a law-checker

In [2]:
from itertools import product

class FiniteCategory:
    """objects, arrows (name -> (dom, cod)), a composition table,
       and a chosen identity arrow for each object."""
    def __init__(self, objects, arrows, comp, ident):
        self.obj, self.ar, self.comp, self.id = set(objects), arrows, comp, ident

    def dom(self, a): return self.ar[a][0]
    def cod(self, a): return self.ar[a][1]
    def composable(self, g, f): return self.dom(g) == self.cod(f)

    def check(self):
        for X in self.obj:                              # identities typed X -> X
            assert self.ar[self.id[X]] == (X, X)
        for g, f in product(self.ar, repeat=2):         # composition total & typed
            if self.composable(g, f):
                gf = self.comp[(g, f)]
                assert self.ar[gf] == (self.dom(f), self.cod(g))
        for f in self.ar:                               # unit laws
            X, Y = self.ar[f]
            assert self.comp[(self.id[Y], f)] == f
            assert self.comp[(f, self.id[X])] == f
        for h, g, f in product(self.ar, repeat=3):      # associativity
            if self.composable(h, g) and self.composable(g, f):
                assert self.comp[(h, self.comp[(g, f)])] == \
                       self.comp[(self.comp[(h, g)], f)]
        return True

# A monoid as a ONE-object category: integers mod 3 under addition.
arrows = {n: ('*', '*') for n in range(3)}            # the arrows ARE the elements
comp   = {(a, b): (a + b) % 3 for a in range(3) for b in range(3)}
Z3 = FiniteCategory({'*'}, arrows, comp, ident={'*': 0})
print(Z3.check())

True


**`broken_table.py`** — a table that is not a category

In [3]:
tbl = {('e','e'):'e', ('e','a'):'a', ('e','b'):'b',
       ('a','e'):'a', ('a','a'):'b', ('a','b'):'b',
       ('b','e'):'b', ('b','a'):'a', ('b','b'):'a'}
Bad = FiniteCategory({'*'}, {x: ('*', '*') for x in 'eab'}, tbl, {'*': 'e'})
try:
    Bad.check()
except AssertionError:
    print("rejected: associativity fails")

fails = [(h, g, f) for h in 'ab' for g in 'ab' for f in 'ab'
         if tbl[(h, tbl[(g, f)])] != tbl[(tbl[(h, g)], f)]]
print(len(fails), "of 8 non-identity triples fail")

rejected: associativity fails
8 of 8 non-identity triples fail


## Exercise 12

**`three.py`**

In [4]:
objs = {0, 1, 2}
ar = {'i0':(0,0), 'i1':(1,1), 'i2':(2,2),
      'a':(0,1), 'b':(1,2), 'c':(0,2)}        # c is forced:  b . a
ident = {0:'i0', 1:'i1', 2:'i2'}

# at most one arrow per ordered pair, so each composite is determined
def only(s, t):
    return next(n for n,(d,c) in ar.items() if d==s and c==t)
comp = {(g,f): only(ar[f][0], ar[g][1])
        for g in ar for f in ar if ar[f][1]==ar[g][0]}

print(FiniteCategory(objs, ar, comp, ident).check())   # True

True


## Exercise 13

**`corrupt.py`** — the checker must object

In [5]:
bad_comp = dict(comp)            # comp: the table of THREE (Exercise 12)
bad_comp[('b', 'a')] = 'i0'      # claim: 0 -> 1 -> 2 composes to id_0
try:
    FiniteCategory(objs, ar, bad_comp, ident).check()
except AssertionError:
    print("rejected: b . a must be an arrow 0 -> 2")

rejected: b . a must be an arrow 0 -> 2
